In [14]:
import re
import os
from tqdm import tqdm
from collections import defaultdict, Counter

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pydub import AudioSegment
import opensmile

from transformers import AutoTokenizer, AutoModel, BertForSequenceClassification
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, LinearLR
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import compute_class_weight

import shap
from enum import Enum
import umap.umap_ as umap
from imblearn.over_sampling import SMOTE

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# TODO:
- New multi-branch model:
  - Instead of embeddings branch use BERT itself (bertforsequenceclassification)
  - How do I scale audio? All together with StandardScaler ~~Each feature separately? Group them somehow? All together?~~
- SHAP
- 10-fold CV

## Possible alterations:
- Try artificialy oversampling data to get 50/50 split of control/dementia (I have a big skew towards dementia samples, current workaround: calculating weights for the loss function)
- Try adding quantitative speech markers (like speech rate and filler words count)
- Maybe fuck .cha transcripts? Use them to get utterances timestamps and then try Whisper to get Speech-To-Text
- Try machine translation to RU

In [3]:
df = pd.read_pickle('/home/gabs/Documents/lab-NN/cognitive_impairments/data/DementiaBank/parsed_data.pkl')

In [4]:
df['dementia'].unique()

array(['PossibleAD', 'ProbableAD', 'Memory', 'Vascular', 'MCI', 'Other',
       'Probable', 'Control'], dtype=object)

In [5]:
# Encode values
SIMPLE = True

sex_mapping = {'female': 0,
               'male': 1}
dementia_mapping = {'Control' : 0,
                    'PossibleAD' : 1,
                    'ProbableAD' : 1,
                    'Probable' : 1,
                    'Memory' : 2,
                    'Vascular' : 3,
                    'MCI' : 4,
                    'Other' : 5}

simple_dementia_mapping = {'Control' : 0,
                    'PossibleAD' : 1,
                    'ProbableAD' : 1,
                    'Probable' : 1,
                    'Memory' : 1,
                    'Vascular' : 1,
                    'MCI' : 1,
                    'Other' : 1}
df['sex'] = df['sex'].map(sex_mapping)

# I suppose we can keep simple mapping, cuz 'PossibleAD' and 'ProbableAD' are the most common (like 80% of the data)
if SIMPLE:
    df['dementia'] = df['dementia'].map(simple_dementia_mapping)
else:
    df['dementia'] = df['dementia'].map(dementia_mapping)

In [6]:
print(f"cookie stats\n{df[df['ID'].str.contains(r'cookie$', regex=True)].groupby('dementia').size()}\n")
print(f"recall stats\n{df[df['ID'].str.contains(r'recall$', regex=True)].groupby('dementia').size()}\n")
print(f"fluency stats\n{df[df['ID'].str.contains(r'fluency$', regex=True)].groupby('dementia').size()}\n")
print(f"sentence stats\n{df[df['ID'].str.contains(r'sentence$', regex=True)].groupby('dementia').size()}\n")

cookie stats
dementia
0    243
1    306
dtype: int64

recall stats
dementia
0      1
1    238
dtype: int64

fluency stats
dementia
0      1
1    210
dtype: int64

sentence stats
dementia
0      1
1    213
dtype: int64



In [7]:
JUST_COOKIE = False
# extract cookie data, cuz every other task has only 1 control sample
if JUST_COOKIE:
    df = df[df['ID'].str.contains(r'cookie$', regex=True)]

In [8]:
df.groupby('dementia').size()

dementia
0    246
1    967
dtype: int64

In [9]:
df


,ID,age,sex,dementia,text,embeddings,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,476-0_cookie,64,0,1,clearsthroat hm the woman of the house is dryi...,"[[-0.0019944734, 0.11227388, 0.29719827, -0.04...",32.367973,0.169069,28.859549,32.672092,...,0.014956,-0.010772,0.069282,4.552497,3.578753,0.414964,0.119592,0.077508,0.064468,-29.202118
1,183-1_cookie,64,1,1,the boy's uh fallin off the stool the the girl...,"[[-0.023478348, 0.15287554, 0.18057655, -0.068...",25.893515,0.083554,25.068998,26.007025,...,0.019386,-0.003246,0.022419,2.437523,1.581530,0.449667,0.280772,0.284074,0.188013,-31.610138
2,216-0_cookie,72,0,1,well there's a kid in a cookie jar there's a g...,"[[-0.17489381, 0.079453476, 0.30083305, 0.0173...",30.133780,0.158424,26.081902,31.247602,...,0.036977,-0.013100,0.038447,3.465397,2.478610,0.336407,0.197328,0.126481,0.055212,-37.809464
3,173-1_cookie,71,0,1,let's see what's this this well the k lu and t...,"[[-0.08635035, 0.13704914, 0.41061246, -0.0872...",29.214741,0.193511,25.452457,28.852304,...,0.038325,-0.020275,0.022480,3.307396,2.552122,0.367143,0.228349,0.163214,0.070129,-46.272060
4,381-0_cookie,75,0,1,well the kids are swipin the cookies mom's doi...,"[[0.22607478, 0.18136616, 0.2866592, -0.016044...",27.629732,0.074770,26.195204,28.142347,...,0.037889,-0.009598,0.008773,3.657219,2.840905,0.122197,0.068713,0.360424,0.285403,-53.288258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,128-1_cookie,71,0,0,oh I remember this one okay the mom's doing th...,"[[-0.33195722, 0.024904918, 0.24865773, 0.0070...",30.967085,0.103636,28.283186,30.838778,...,0.054463,-0.014791,0.021959,3.372949,2.116813,0.310652,0.159613,0.214994,0.122742,-45.492466
1209,142-3_cookie,61,0,0,mhm well the water's running over on the floor...,"[[-0.1724466, -0.102807336, 0.31384772, 0.0262...",33.519424,0.088880,32.100095,33.175519,...,0.056380,-0.012752,0.024856,4.028542,1.806884,0.417500,0.195181,0.095458,0.067621,-29.882867
1210,332-0_fluency,50,1,0,dog cow cat horse sheep mule donkey deer antel...,"[[0.073371045, 0.3103367, -0.15680264, -0.1120...",26.910131,0.028706,26.678370,27.039707,...,-0.001642,-0.000825,0.011415,1.168898,0.703156,0.265128,0.025641,1.202970,0.899072,-54.753613
1211,332-0_recall,50,1,0,eorge and his granddaughter went to visit ( ) ...,"[[-0.32294938, -0.06343152, 0.38732597, -0.025...",25.061729,0.053765,24.179365,24.968309,...,0.002817,-0.002691,0.013942,3.129667,1.758500,0.162454,0.099134,0.416941,0.612056,-54.896027


# Train model

In [31]:
class PittDataset(Dataset):
    def __init__(self, texts, audio_features, age, sex, labels, max_length=128):
        self.texts = texts
        self.audio_features = audio_features
        self.age = age.reshape(-1, 1)
        self.sex = sex.reshape(-1, 1)
        self.labels = labels
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])  # Ensure text is string
        audio = self.audio_features[idx]
        age = self.age[idx]
        sex = self.sex[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'audio_features': torch.tensor(audio, dtype=torch.float32),
            'age': torch.tensor(age, dtype=torch.float32).view(-1),  # Ensure proper shape
            'sex': torch.tensor(sex, dtype=torch.float32).view(-1),   # Ensure proper shape
            'label': torch.tensor(label, dtype=torch.long)
        }

In [33]:
embeddings = df['text'].values
audio_features = df.drop(columns=['ID', 'embeddings', 'dementia', 'age', 'sex', 'text']).values
age = df['age'].values
sex = df['sex'].values
labels = df['dementia'].values
'''
 there's some fuckery with embeddings, df['embeddings'].values returns an array with arrays inside with the shape of (n_samples,)
 it's like a list with these arrays, but we need to glue them together to get their real shape, that's why we use np.stack()
 however, it returns the needed array with the shape of (n_samples, 1, 768), and that's why we squeeze it
 as a result we get the shape of (n_samples, 768)
'''

X_emb_train, X_emb_test, X_audio_train, X_audio_test, X_age_train, X_age_test, X_sex_train, X_sex_test, y_train, y_test = train_test_split(
    embeddings, 
    audio_features, 
    age, 
    sex, 
    labels, test_size=0.2, stratify=df['dementia'])

train_dataset = PittDataset(X_emb_train, X_audio_train, X_age_train, X_sex_train, y_train)
test_dataset = PittDataset(X_emb_test, X_audio_test, X_age_test, X_sex_test, y_test)


In [34]:
print("Scaled audio features stats:")
print(f"Min: {train_dataset.audio_features.min()}, Max: {train_dataset.audio_features.max()}")
print(f"Mean: {train_dataset.audio_features.mean()}, Std: {train_dataset.audio_features.std()}")
train_dataset.audio_features.shape

Scaled audio features stats:
Min: -2686.986083984375, Max: 3075.130859375
Mean: 94.59817678833544, Std: 372.6651255230868


(970, 88)

In [35]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2  # Binary classification
)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn

def run_epochs(model, train_loader, val_loader, device, num_epochs=10, lr=2e-5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, 
                                              num_warmup_steps=0,
                                              num_training_steps=total_steps)
    
    criterion = nn.CrossEntropyLoss()
    
    best_val_f1 = 0
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, 
                           attention_mask=attention_mask, 
                           labels=labels)
            
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        
        model.eval()
        val_preds = []
        val_true = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                
                preds = torch.argmax(logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        
        train_loss = total_loss / len(train_loader)
        val_acc = accuracy_score(val_true, val_preds)
        val_f1 = f1_score(val_true, val_preds)
        
        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
    
    print(f'Best Validation F1: {best_val_f1:.4f}')

In [37]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

run_epochs(model, train_loader, test_loader, device)

Epoch 1/10
Train Loss: 0.8673 | Val Acc: 0.2058 | Val F1: 0.0103
Epoch 2/10
Train Loss: 0.8666 | Val Acc: 0.2058 | Val F1: 0.0103
Epoch 3/10
Train Loss: 0.8543 | Val Acc: 0.2058 | Val F1: 0.0103
Epoch 4/10
Train Loss: 0.8547 | Val Acc: 0.2058 | Val F1: 0.0103
Epoch 5/10
Train Loss: 0.8489 | Val Acc: 0.2058 | Val F1: 0.0103
Epoch 6/10
Train Loss: 0.8462 | Val Acc: 0.2058 | Val F1: 0.0103


KeyboardInterrupt: 